# 🌲 PINE 2기 7회차 — AI 에이전트 개발 2차

> **유스AI프로젝트:D** | 마포청소년문화의집 × KB데이타시스템 × Microsoft  
> 사용 기술: **Microsoft Agent Framework (MAF) 1.0.0** + Azure OpenAI (API Key)

---

## 🗓 오늘의 여정 (180분)

```
[10:00~10:15] ⚙️  환경 설정 & 복습
[10:15~10:40] 🔗 Part 1: Sequential — 순차 에이전트 파이프라인
              └─ 🔴 Quiz 1: 번역 파이프라인 만들기 (5분)
[10:40~11:05] ⚡ Part 2: Concurrent — 병렬 에이전트
              └─ 🔴 Quiz 2: 동시 분석가 팀 만들기 (5분)
[11:05~11:40] 💬 Part 3: GroupChat — 역할 에이전트 토론
              └─ 🔴 Quiz 3: 나만의 GroupChat 설계 (10분)
[11:40~12:05] 🧠 Part 4: Session — 대화 메모리
              └─ 🔴 Quiz 4: 기억하는 공부 플래너 (10분)
[12:05~12:45] 📚 Part 5: RAG — 외부 지식 검색 에이전트
              └─ 🔴 Quiz 5: RAG 진로상담 에이전트 완성 (15분)
[12:45~13:00] 🎤 팀 발표 & Q&A & 해커톤 예고
```

## 📌 지난 회차(6회차) 복습
- ✅ `Agent` + `OpenAIChatClient` 기본 생성
- ✅ `@tool` 데코레이터로 도구 등록
- ✅ ReAct 패턴 (Reason → Act → Observe)
- ✅ 수학 문제 해결 에이전트

**오늘**: 에이전트 여러 개를 **연결**하고, **기억**하게 하고, **검색**까지 시킵니다! 🚀

---
# ⚙️ 섹션 0: 환경 설정

> **주의**: 이 셀들을 반드시 먼저 순서대로 실행하세요!  
> Jupyter에서는 `await`를 셀에서 바로 사용할 수 있습니다.

In [ ]:
# 설치 확인 (Codespace는 자동 설치, 로컬이면 아래 주석 해제)
# !pip install agent-framework==1.0.0 python-dotenv scikit-learn numpy -q

In [ ]:
# =============================================
# 📦 임포트 — MAF 1.0.0 공식 API
# =============================================
import os
from typing import cast
from dotenv import load_dotenv

# MAF 핵심
from agent_framework import Agent, tool, Message
from agent_framework.openai import OpenAIChatClient

# MAF 오케스트레이션 패턴 (4종)
from agent_framework.orchestrations import (
    SequentialBuilder,   # 순차: A → B → C
    ConcurrentBuilder,   # 병렬: A & B & C 동시
    GroupChatBuilder,    # 그룹챗: 역할 에이전트 토론
    HandoffBuilder,      # 핸드오프: 전문가 라우팅
)
from agent_framework_orchestrations import GroupChatState  # GroupChat 상태 타입

load_dotenv()

import importlib.metadata
print("✅ 임포트 성공!")
print(f"   agent-framework: {importlib.metadata.version('agent-framework')}")

In [ ]:
# =============================================
# 🔑 Azure OpenAI 클라이언트 초기화
# =============================================
# OpenAIChatClient는 아래 환경변수를 자동으로 읽습니다:
#   AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, AZURE_OPENAI_MODEL

client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_MODEL"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview"),
)
print(f"✅ 클라이언트 준비! 모델: {os.environ['AZURE_OPENAI_MODEL']}")

In [ ]:
# =============================================
# 🧪 연결 테스트
# =============================================
test_agent = Agent(
    client=client,
    name="테스트봇",
    instructions="당신은 친절한 AI입니다."
)

# agent.run()은 비동기 → await 필요
# 반환값은 AgentResponse 객체 → .text 로 텍스트 추출
response = await test_agent.run("한 문장으로 자기소개 해줘!")
print("🤖", response.text)

In [ ]:
# =============================================
# 🛠 공통 유틸: 워크플로우 실행 & 출력 헬퍼
# =============================================
# workflow.run()의 이벤트 타입:
#   "data"   → AgentResponseUpdate (스트리밍 중간 토큰)
#   "output" → list[Message]        (최종 결과)

from agent_framework import WorkflowEvent
from agent_framework._types import AgentResponseUpdate

async def run_workflow(workflow, task: str, verbose: bool = True) -> list:
    """워크플로우를 실행하고 최종 메시지 리스트를 반환합니다."""
    if verbose:
        print(f"📢 과제: {task}")
        print("=" * 60)

    final_messages = []
    current_author = None

    async for event in workflow.run(task, stream=True):
        if event.type == "data" and isinstance(event.data, AgentResponseUpdate):
            # 스트리밍: 에이전트가 토큰을 생성하는 중
            author = event.data.author_name or event.executor_id or "?"
            if author != current_author:
                if current_author is not None:
                    print()  # 이전 에이전트 줄바꿈
                print(f"\n🤖 [{author}]: ", end="", flush=True)
                current_author = author
            if verbose:
                print(event.data.text, end="", flush=True)

        elif event.type == "output":
            # 최종 결과
            data = event.data
            if isinstance(data, list):
                final_messages = data

    if verbose:
        print(f"\n\n✅ 완료 | 총 {len(final_messages)}개 메시지")
    return final_messages

print("✅ 헬퍼 함수 준비!")

---
# 🔗 Part 1: Sequential — 순차 에이전트 파이프라인

## 개념: 에이전트가 컨베이어 벨트처럼 연결된다

```
입력
 │
 ▼
[에이전트 A: 작가]  ← 초안 작성
 │  (A의 출력이 B의 입력)
 ▼
[에이전트 B: 편집자] ← 퀄리티 향상
 │  (B의 출력이 C의 입력)
 ▼
[에이전트 C: 검토자] ← 최종 확인
 │
 ▼
최종 결과
```

각 에이전트는 이전 에이전트의 **전체 대화 이력**을 볼 수 있습니다.

## MAF 1.0.0 API
```python
from agent_framework.orchestrations import SequentialBuilder

workflow = SequentialBuilder(participants=[agentA, agentB, agentC]).build()
async for event in workflow.run("과제", stream=True): ...
```

In [ ]:
# =============================================
# 🔗 예제: 블로그 글 작성 파이프라인
# =============================================

# 에이전트 1: 작가 — 초안 작성
writer = Agent(
    client=client,
    name="작가",
    instructions="""당신은 창의적인 블로그 작가입니다.
주어진 주제로 흥미롭고 읽기 쉬운 블로그 글 초안을 3~4문장으로 작성하세요.
고등학생 독자를 대상으로 합니다."""
)

# 에이전트 2: 편집자 — 내용 개선
editor = Agent(
    client=client,
    name="편집자",
    instructions="""당신은 경험 많은 편집자입니다.
앞의 글을 읽고, 더 명확하고 흥미롭게 개선하세요.
불필요한 부분은 삭제하고, 부족한 내용은 보완하세요. 3~4문장 유지."""
)

# 에이전트 3: 검토자 — 최종 평가
reviewer = Agent(
    client=client,
    name="검토자",
    instructions="""당신은 까다로운 콘텐츠 검토자입니다.
지금까지의 글을 평가하고 최종 개선점을 2~3줄로 간략히 제안하세요.
반드시 '최종 평가:'로 시작하세요."""
)

# SequentialBuilder로 파이프라인 구성
blog_pipeline = SequentialBuilder(
    participants=[writer, editor, reviewer]  # 이 순서대로 실행
).build()

print("✅ 블로그 파이프라인 구성 완료! (작가 → 편집자 → 검토자)")

In [ ]:
# 파이프라인 실행
blog_messages = await run_workflow(
    blog_pipeline,
    "AI 에이전트가 미래 직업에 미치는 영향"
)

# 최종 결과 확인
print("\n📝 최종 메시지 목록:")
for msg in blog_messages:
    if msg.author_name and msg.role == "assistant":
        print(f"  [{msg.author_name}]: {msg.text[:80]}...")

---
## 🔴 Quiz 1: 번역 파이프라인 만들기

### 📋 목표
**영어 → 일본어 → 한국어** 순서로 번역하는 3단계 Sequential 파이프라인을 만드세요.

### ⏱ 제한시간: 5분

### 성공 기준
- 3개 번역 에이전트가 순서대로 실행된다
- 각 단계의 출력이 다음 에이전트 입력으로 전달된다

In [ ]:
# ============================================================
# 🔴 Quiz 1: 번역 파이프라인 (한국어 → 영어 → 일본어 → 다시 한국어)
# ============================================================
# TODO: ??? 부분을 채워서 완성하세요! (5분)
# ============================================================

# 1️⃣ 영어 번역 에이전트
to_english = Agent(
    client=client,
    name="영어번역가",
    instructions="???"  # 힌트: 입력을 영어로 번역, 번역 결과만 출력
)

# 2️⃣ 일본어 번역 에이전트
to_japanese = Agent(
    client=client,
    name="일본어번역가",
    instructions="???"  # 힌트: 앞의 영어 텍스트를 일본어로 번역
)

# 3️⃣ 역번역 에이전트 (일본어 → 한국어)
back_to_korean = Agent(
    client=client,
    name="역번역가",
    instructions="???"  # 힌트: 일본어를 한국어로 역번역, 원문과 비교 코멘트 한 줄 추가
)

# 4️⃣ SequentialBuilder로 파이프라인 구성
translation_pipeline = SequentialBuilder(
    participants=[???, ???, ???]
).build()

# 5️⃣ 실행!
quiz1_result = await run_workflow(
    translation_pipeline,
    "인공지능은 우리의 일상을 더 편리하게 만들어준다."
)

In [ ]:
# ✅ Quiz 1 정답 (강사 참고용)
# to_english = Agent(client=client, name="영어번역가",
#     instructions="입력된 한국어를 자연스러운 영어로 번역하세요. 번역 결과만 출력합니다.")
# to_japanese = Agent(client=client, name="일본어번역가",
#     instructions="이전 메시지의 영어를 자연스러운 일본어로 번역하세요. 번역 결과만 출력합니다.")
# back_to_korean = Agent(client=client, name="역번역가",
#     instructions="이전 메시지의 일본어를 한국어로 역번역하세요. 마지막에 '원문과의 차이: ' 한 줄을 추가하세요.")
# translation_pipeline = SequentialBuilder(participants=[to_english, to_japanese, back_to_korean]).build()
# quiz1_result = await run_workflow(translation_pipeline, "인공지능은 우리의 일상을 더 편리하게 만들어준다.")

---
# ⚡ Part 2: Concurrent — 병렬 에이전트

## 개념: 여러 에이전트가 동시에 다른 각도로 분석한다

```
         입력
          │
    ┌─────┼─────┐
    ▼     ▼     ▼     ← 동시 실행!
 [전문가A] [전문가B] [전문가C]
    └─────┼─────┘
          ▼
    결과 합산 (aggregation)
```

Sequential과의 차이: A가 B를 기다리지 않음 → **속도 ↑**

## MAF 1.0.0 API
```python
from agent_framework.orchestrations import ConcurrentBuilder

workflow = ConcurrentBuilder(
    participants=[agentA, agentB, agentC]
).build()
```

In [ ]:
# =============================================
# ⚡ 예제: AI 기술 트렌드 동시 분석
# =============================================

# 3명의 전문가가 같은 주제를 동시에 각자의 관점으로 분석
tech_analyst = Agent(
    client=client,
    name="기술분석가",
    instructions="""당신은 AI 기술 전문가입니다.
주어진 주제의 기술적 측면(알고리즘, 인프라, 성능)을 3~4문장으로 분석하세요.
반드시 '[기술 분석]'으로 시작하세요."""
)

market_analyst = Agent(
    client=client,
    name="시장분석가",
    instructions="""당신은 IT 시장 전문가입니다.
주어진 주제의 시장/비즈니스 영향(수요, 일자리, 투자)을 3~4문장으로 분석하세요.
반드시 '[시장 분석]'으로 시작하세요."""
)

ethics_analyst = Agent(
    client=client,
    name="윤리분석가",
    instructions="""당신은 AI 윤리 전문가입니다.
주어진 주제의 윤리적 영향(개인정보, 공정성, 사회 영향)을 3~4문장으로 분석하세요.
반드시 '[윤리 분석]'으로 시작하세요."""
)

# ConcurrentBuilder: 세 에이전트가 동시에 실행!
trend_analysis = ConcurrentBuilder(
    participants=[tech_analyst, market_analyst, ethics_analyst]
).build()

print("✅ 동시 분석 워크플로우 구성 완료! (3명이 동시에 분석)")

In [ ]:
import time

# 병렬 실행
start = time.time()
concurrent_messages = await run_workflow(
    trend_analysis,
    "멀티 에이전트 AI 시스템이 고등학생의 학습에 미치는 영향"
)
elapsed = time.time() - start
print(f"\n⏱ 소요 시간: {elapsed:.1f}초 (3명이 동시 실행)")
print("  → Sequential이었다면 약 3배 더 걸렸을 것!")

---
## 🔴 Quiz 2: 동시 분석가 팀 만들기

### 📋 목표
**수학**, **과학**, **영어** 3명의 과외 선생님이 동시에 "AI 시대 필수 공부법"을 분석하는
ConcurrentBuilder 워크플로우를 만드세요.

### ⏱ 제한시간: 5분

In [ ]:
# ============================================================
# 🔴 Quiz 2: 동시 분석가 팀
# ============================================================
# TODO: ??? 부분을 채워서 완성하세요! (5분)
# ============================================================

# 1️⃣ 수학 선생님
math_teacher = Agent(
    client=client,
    name="수학선생님",
    instructions="???"  # [수학 관점]으로 시작, 3문장
)

# 2️⃣ 과학 선생님
science_teacher = Agent(
    client=client,
    name="과학선생님",
    instructions="???"  # [과학 관점]으로 시작, 3문장
)

# 3️⃣ 영어 선생님
english_teacher = Agent(
    client=client,
    name="영어선생님",
    instructions="???"  # [영어 관점]으로 시작, 3문장
)

# 4️⃣ ConcurrentBuilder로 구성
teacher_team = ConcurrentBuilder(
    participants=[???, ???, ???]
).build()

# 5️⃣ 실행!
quiz2_result = await run_workflow(
    teacher_team,
    "AI 시대에 고등학생이 반드시 공부해야 할 것들"
)

In [ ]:
# ✅ Quiz 2 정답 (강사 참고용)
# math_teacher = Agent(client=client, name="수학선생님",
#     instructions="당신은 수학 선생님입니다. '[수학 관점]'으로 시작해 AI 시대 수학 공부의 중요성을 3문장으로 말하세요.")
# science_teacher = Agent(client=client, name="과학선생님",
#     instructions="당신은 과학 선생님입니다. '[과학 관점]'으로 시작해 AI 시대 과학 공부의 중요성을 3문장으로 말하세요.")
# english_teacher = Agent(client=client, name="영어선생님",
#     instructions="당신은 영어 선생님입니다. '[영어 관점]'으로 시작해 AI 시대 영어 공부의 중요성을 3문장으로 말하세요.")
# teacher_team = ConcurrentBuilder(participants=[math_teacher, science_teacher, english_teacher]).build()
# quiz2_result = await run_workflow(teacher_team, "AI 시대에 고등학생이 반드시 공부해야 할 것들")

---
# 💬 Part 3: GroupChat — 역할 에이전트 토론

## 개념: 에이전트들이 서로의 발언을 듣고 대화한다

```
       ┌──────────────────────┐
       │   selection_func()   │ ← 누가 다음에 말할지 결정
       └──────┬───────────────┘
              │
     ┌────────┼────────┐
     ▼        ▼        ▼
 [학생]   [전문가]  [비평가]  ← 서로의 발언을 보며 대화
```

## MAF 1.0.0 정확한 API

```python
from agent_framework.orchestrations import GroupChatBuilder
from agent_framework_orchestrations import GroupChatState

# selection_func: GroupChatState를 받아 다음 에이전트 이름 반환
def my_selector(state: GroupChatState) -> str | None:
    if state.current_round >= 6:     # ← state.current_round (int)
        return None                  # ← None = 종료
    names = list(state.participants.keys())  # ← state.participants (OrderedDict)
    return names[state.current_round % len(names)]

# GroupChatBuilder: 생성자 파라미터로 전달!
# (주의: .set_select_speakers_func().participants() 체이닝 방식 ❌)
workflow = GroupChatBuilder(
    participants=[agent1, agent2, agent3],
    selection_func=my_selector,
    max_rounds=6,
).build()
```

In [ ]:
# =============================================
# 💬 예제: 선생님-학생 GroupChat
# =============================================

gc_teacher = Agent(
    client=client,
    name="선생님",
    instructions="""당신은 친절한 AI 선생님입니다.
개념을 쉽게 설명하고 학생의 이해를 확인하는 질문을 던집니다. 3문장."""
)

gc_student = Agent(
    client=client,
    name="학생",
    instructions="""당신은 호기심 많은 고등학생입니다.
모르면 솔직하게 질문하고, 이해한 내용을 자기 말로 표현합니다. 2문장."""
)

# 라운드로빈 발언자 선택 함수 (MAF 1.0.0 정확한 API)
def round_robin(state: GroupChatState) -> str | None:
    if state.current_round >= 4:          # 4라운드 후 종료
        return None
    names = list(state.participants.keys())
    return names[state.current_round % len(names)]  # 번갈아가며

# GroupChatBuilder — 생성자에 파라미터 전달!
ts_chat = GroupChatBuilder(
    participants=[gc_teacher, gc_student],
    selection_func=round_robin,
).build()

print("✅ GroupChat 구성 완료!")

In [ ]:
# GroupChat 실행
gc_messages = await run_workflow(
    ts_chat,
    "멀티 에이전트 AI가 무엇인지 설명해주세요!"
)

In [ ]:
# =============================================
# 💡 심화: 조건부 발언자 선택
# =============================================
# 마지막 발언자의 이름에 따라 다음 발언자를 결정!

expert = Agent(
    client=client,
    name="전문가",
    instructions="AI 기술을 정확하고 상세하게 설명합니다. 4문장."
)
student2 = Agent(
    client=client,
    name="학생",
    instructions="모르는 것을 솔직하게 질문합니다. 2문장."
)
critic = Agent(
    client=client,
    name="비평가",
    instructions="앞선 설명의 허점을 지적하고 개선 방향을 제시합니다. 3문장."
)

# 스마트 발언자 선택: 비평가가 마지막에 항상 마무리
def smart_selector(state: GroupChatState) -> str | None:
    if state.current_round >= 6:
        return None

    names = list(state.participants.keys())

    # 마지막 발언자 확인
    if state.conversation:
        last_msg = state.conversation[-1]
        last_author = last_msg.author_name
        if last_author == "전문가":
            return "학생"   # 전문가 → 학생이 질문
        elif last_author == "학생":
            return "전문가" # 학생 → 전문가가 답변

    # 첫 발언 또는 기타: 라운드로빈
    return names[state.current_round % len(names)]

smart_chat = GroupChatBuilder(
    participants=[expert, student2, critic],
    selection_func=smart_selector,
    max_rounds=6,
).build()

smart_messages = await run_workflow(
    smart_chat,
    "RAG 에이전트는 왜 필요한가요?"
)

---
## 🔴 Quiz 3: 나만의 GroupChat 설계

### 📋 목표
**찬성론자**, **반대론자**, **중재자** 3개 에이전트로  
"AI가 교사를 대체할 수 있을까?" 주제로 GroupChat 토론을 실행하세요.

### ⏱ 제한시간: 10분

### 핵심 체크포인트
- `GroupChatBuilder(participants=[...], selection_func=fn)` — 생성자 파라미터 방식!
- `state.current_round` (int), `state.participants` (OrderedDict)

In [ ]:
# ============================================================
# 🔴 Quiz 3: 나만의 GroupChat 토론
# ============================================================
# TODO: ??? 부분을 채워서 완성하세요! (10분)
# ============================================================

# 1️⃣ 찬성론자
supporter = Agent(
    client=client,
    name="찬성론자",
    instructions="???"  # AI가 교사를 대체할 수 있다는 주장, 3문장
)

# 2️⃣ 반대론자
opposer = Agent(
    client=client,
    name="반대론자",
    instructions="???"  # AI가 교사를 대체할 수 없다는 주장, 3문장
)

# 3️⃣ 중재자
moderator = Agent(
    client=client,
    name="중재자",
    instructions="???"  # 양측 의견을 정리하고 균형잡힌 시각 제시, 3문장
)

# 4️⃣ 발언자 선택 함수 (찬성 → 반대 → 중재 → 반복)
def debate_selector(state: GroupChatState) -> str | None:
    if state.current_round >= ???:  # 몇 라운드?
        return None
    names = list(state.???.keys())  # state의 어떤 필드?
    return names[state.??? % len(names)]  # 현재 라운드는?

# 5️⃣ GroupChatBuilder 구성 (생성자 파라미터!)
debate_chat = GroupChatBuilder(
    participants=[???, ???, ???],
    selection_func=???,
).build()

# 6️⃣ 토론 시작!
debate_messages = await run_workflow(
    debate_chat,
    "주제: AI가 학교 교사를 완전히 대체할 수 있을까? 각자의 입장에서 토론하세요."
)

In [ ]:
# ✅ Quiz 3 정답 (강사 참고용)
# supporter = Agent(client=client, name="찬성론자",
#     instructions="AI가 교사를 대체할 수 있다고 강하게 주장합니다. 구체적 근거 포함. 3문장.")
# opposer = Agent(client=client, name="반대론자",
#     instructions="AI가 교사를 대체할 수 없다고 강하게 주장합니다. 인간 고유의 가치 강조. 3문장.")
# moderator = Agent(client=client, name="중재자",
#     instructions="양측 주장의 핵심을 정리하고 균형잡힌 결론을 제시합니다. 3문장.")
# def debate_selector(state: GroupChatState) -> str | None:
#     if state.current_round >= 6: return None
#     names = list(state.participants.keys())
#     return names[state.current_round % len(names)]
# debate_chat = GroupChatBuilder(participants=[supporter, opposer, moderator], selection_func=debate_selector).build()
# debate_messages = await run_workflow(debate_chat, "주제: AI가 학교 교사를 완전히 대체할 수 있을까?")

---
# 🧠 Part 4: Session — 대화 메모리

## 개념: 에이전트가 이전 대화를 기억한다

```
세션 없이:
  [호출 1] "내 이름은 지민이야" → "안녕, 지민님!"
  [호출 2] "내 이름이 뭐야?"   → "모릅니다..." 😅

세션 있이:
  session = agent.create_session()    ← 동기 함수! await 없음
  [호출 1] run("지민이야", session=session) → "안녕, 지민님!"
  [호출 2] run("이름이 뭐야?", session=session) → "지민님이라고 하셨죠!" ✅
```

## MAF 1.0.0 핵심 주의사항

```python
# ✅ 올바른 방법: create_session()은 동기 함수!
session = agent.create_session()      # await 없음!
response = await agent.run("...", session=session)
print(response.text)                  # .text로 텍스트 추출

# ❌ 틀린 방법
session = await agent.create_session()  # await 하면 오류!
```

In [ ]:
# =============================================
# 🧠 세션 없이 vs 세션 있이 비교 실험
# =============================================

memory_agent = Agent(
    client=client,
    name="기억봇",
    instructions="당신은 학생의 정보를 기억하고 맞춤형 도움을 주는 AI 학습 도우미입니다."
)

print("=" * 50)
print("❌ 세션 없이 — 매번 새로 시작")
print("=" * 50)
await memory_agent.run("안녕! 나는 고2 민준이야. 수학이 약해.")
r_no = await memory_agent.run("내 이름이 뭐야?")
print("🤖", r_no.text)

print()
print("=" * 50)
print("✅ 세션 있이 — 대화 이력 유지!")
print("=" * 50)

# create_session()은 동기 함수 — await 없음!
my_session = memory_agent.create_session()

await memory_agent.run("안녕! 나는 고2 민준이야. 수학이 약해.", session=my_session)
r_yes = await memory_agent.run("내 이름이 뭐야?", session=my_session)
print("🤖", r_yes.text)

In [ ]:
# =============================================
# 🧠 멀티턴 세션 예제: 공부 플래너
# =============================================

planner = Agent(
    client=client,
    name="공부플래너",
    instructions="""당신은 고등학생 전문 공부 플래너 AI입니다.
학생의 이름, 약한 과목, 시험 날짜, 목표 점수를 파악하고 기억하세요.
매 답변에서 학생의 이름을 부르고, 맞춤형 계획을 제안합니다.
격려하는 말로 마무리하세요."""
)

# 세션 생성 (동기!)
planner_session = planner.create_session()

conversations = [
    "안녕! 나는 고2 서연이야. 수학 기말고사가 3주 후인데 현재 45점이야. 80점이 목표야!",
    "어떤 단원부터 시작하면 좋을까?",
    "내 이름이랑 목표 점수 기억해?",  # ← 기억 확인!
]

print("📓 공부 플래너 대화")
print("=" * 55)

for msg in conversations:
    print(f"\n👤 서연: {msg}")
    response = await planner.run(msg, session=planner_session)
    print(f"🤖 플래너: {response.text}")
    print("-" * 40)

---
## 🔴 Quiz 4: 기억하는 공부 플래너

### 📋 목표
이름·약한 과목·시험 날짜를 기억하는 **나만의 공부 플래너**를  
`create_session()`을 사용해 완성하세요.

### ⏱ 제한시간: 10분

### 핵심 체크포인트
- `create_session()` — **await 없이** 동기 호출!
- `agent.run(msg, session=session)` — session= 파라미터 전달
- `response.text` — 텍스트 추출

In [ ]:
# ============================================================
# 🔴 Quiz 4: 기억하는 공부 플래너
# ============================================================
# TODO: ??? 부분을 채워서 완성하세요! (10분)
# ============================================================

# 1️⃣ 나만의 공부 플래너 에이전트 정의
my_planner = Agent(
    client=client,
    name="나의플래너",
    instructions="""???
    # 포함: 이름/과목/날짜/목표 기억, 이름 불러주기, 구체적 계획, 격려
    """
)

# 2️⃣ 세션 생성 (await 없음!)
quiz4_session = my_planner.???()   # create_session 호출

print("🎓 나의 공부 플래너 시작!")
print("=" * 55)

# 3️⃣ 대화 1: 자기소개 (이름 + 과목 + 시험날짜 포함!)
msg1 = "???"  # 예: "안녕! 나는 고2 ○○야. 영어가 약하고 다음달 기말고사야. 목표 75점!"
r1 = await my_planner.run(msg1, ???=quiz4_session)
print(f"\n👤 나: {msg1}")
print(f"🤖 플래너: {r1.text}")

# 4️⃣ 대화 2: 공부법 질문
msg2 = "???"
r2 = await my_planner.run(msg2, ???=quiz4_session)
print(f"\n👤 나: {msg2}")
print(f"🤖 플래너: {r2.text}")

# 5️⃣ 대화 3: 기억 테스트
r3 = await my_planner.run(
    "내가 어떤 과목이 약하고 시험이 언제라고 했지?",
    ???=quiz4_session
)
print(f"\n👤 나: 내가 어떤 과목이 약하고 시험이 언제라고 했지?")
print(f"🤖 플래너: {r3.text}")

In [ ]:
# ✅ Quiz 4 정답 (강사 참고용)
# my_planner = Agent(client=client, name="나의플래너",
#     instructions="""당신은 고등학생 공부 플래너 AI입니다.
# 학생의 이름, 약한 과목, 시험 날짜, 목표 점수를 반드시 기억하고 활용하세요.
# 매 답변에서 학생의 이름을 불러주고 구체적인 날짜별 계획을 제안합니다.
# 격려의 말로 마무리하세요.""")
# quiz4_session = my_planner.create_session()  # await 없음!
# r1 = await my_planner.run("안녕! 나는 고2 지수야. 영어가 약해서 현재 50점이야. 다음달 3일 기말고사에서 75점이 목표야!", session=quiz4_session)
# print(r1.text)

---
# 📚 Part 5: RAG — 외부 지식 검색 에이전트

## 개념: 검색(Retrieve) + 생성(Generate)

```
일반 LLM:
  질문: "PINE 프로그램이 뭐야?"
  답변: "모릅니다" (학습 데이터에 없음)

RAG 에이전트:
  질문: "PINE 프로그램이 뭐야?"
     ↓  (자동으로 @tool 호출!)
  검색: rag_source.md에서 관련 청크 찾기
     ↓
  답변: "PINE은 마포청소년문화의집과 Microsoft가..." ✅
```

## MAF 1.0.0에서의 RAG 구현

```python
from agent_framework import tool, Agent

@tool(approval_mode="never_require")  # 자동 실행
def search_knowledge_base(query: str) -> str:
    """지식베이스를 검색합니다."""  ← docstring이 에이전트에게 도구 설명!
    # 검색 로직...
    return "검색 결과"

rag_agent = Agent(
    client=client,
    instructions="search_knowledge_base를 사용해 답하세요.",
    tools=[search_knowledge_base]   # 등록!
)
# 에이전트가 필요하다 판단하면 자동으로 도구를 호출합니다
```

In [ ]:
# =============================================
# 📚 Step 1: 지식베이스 로드 & TF-IDF 벡터화
# =============================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def load_chunks(filepath: str, min_len: int = 80) -> list:
    """마크다운 파일을 '---' 기준으로 섹션 분할"""
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    sections = [s.strip() for s in text.split("---") if s.strip()]
    return [s for s in sections if len(s) > min_len]

# 지식베이스 로드
chunks = load_chunks("data/rag_source.md")

# TF-IDF 벡터화 (한국어용 문자 n-gram)
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=5000)
chunk_vectors = vectorizer.fit_transform(chunks)

print(f"✅ {len(chunks)}개 청크 로드 & 벡터화 완료!")
print(f"첫 번째 청크 미리보기: {chunks[0][:80]}...")

In [ ]:
# =============================================
# 📚 Step 2: @tool로 검색 함수 정의
# =============================================

@tool(approval_mode="never_require")
def search_knowledge_base(query: str) -> str:
    """진로, AI 직업, 학습 팁, PINE 프로그램에 관한 지식베이스를 검색합니다.

    Args:
        query: 검색할 내용 (한국어)

    Returns:
        관련 정보 텍스트와 출처 번호
    """
    # 1) 쿼리 벡터화 및 유사도 계산
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, chunk_vectors)[0]

    # 2) Top-3 검색
    top_idx = np.argsort(sims)[::-1][:3]
    results = [(chunks[i], float(sims[i])) for i in top_idx if sims[i] > 0]

    if not results:
        return "관련 정보를 찾지 못했습니다."

    # 3) 결과 포맷
    parts = [
        f"[참고 자료 {i+1}] (관련도: {score:.2f})\n{chunk[:400]}"
        for i, (chunk, score) in enumerate(results)
    ]
    return "\n\n".join(parts)


# 도구 직접 테스트
print("🔍 검색 테스트:")
print(search_knowledge_base("AI 관련 직업")[:300], "...")

In [ ]:
# =============================================
# 📚 Step 3: RAG 에이전트 생성 & 질문
# =============================================

rag_agent = Agent(
    client=client,
    name="진로상담사",
    instructions="""당신은 고등학생의 진로와 학습을 돕는 AI 상담사입니다.
질문에 답할 때 반드시 search_knowledge_base 도구를 사용해 정확한 정보를 검색하세요.
답변 마지막에 반드시 '[출처: 참고 자료 N]' 형식으로 출처를 표시하세요.
참고 자료에 없는 내용은 '추가 확인이 필요합니다'라고 솔직하게 말하세요.""",
    tools=[search_knowledge_base]   # @tool 등록!
)

# 질문 1
q1 = "AI 관련 직업에는 어떤 것들이 있나요?"
print(f"{'='*60}\n❓ {q1}")
a1 = await rag_agent.run(q1)
print(f"🤖 진로상담사:\n{a1.text}")

print()

# 질문 2
q2 = "PINE 프로그램 수료 후 어떤 결과물을 얻을 수 있나요?"
print(f"{'='*60}\n❓ {q2}")
a2 = await rag_agent.run(q2)
print(f"🤖 진로상담사:\n{a2.text}")

In [ ]:
# =============================================
# 📚 심화: RAG + Session 결합!
# =============================================
# RAG로 정확한 정보를 검색하면서 대화 기억도 유지

rag_session = rag_agent.create_session()  # 동기!

print("🌟 RAG + Session 결합 에이전트")
print("=" * 55)

talks = [
    "안녕! 나는 고2 소연이야. AI 연구원이 꿈이야.",
    "AI 연구원이 되려면 어떤 학과에 가야 해?",     # RAG 작동!
    "내 꿈이 뭐라고 했지?",                       # 메모리 확인!
]

for msg in talks:
    print(f"\n👤 소연: {msg}")
    resp = await rag_agent.run(msg, session=rag_session)
    print(f"🤖 상담사: {resp.text}")

---
## 🔴 Quiz 5: RAG 진로상담 에이전트 완성

### 📋 목표
`load_chunks()` → TF-IDF → `@tool` → `Agent` 전체 파이프라인을  
**처음부터 직접 구현**하고, 진로 질문 3개에 답하세요.

### ⏱ 제한시간: 15분

### 성공 기준
- `@tool`로 검색 함수 구현
- 3개 이상 다른 진로 질문에 답변
- 🌟 보너스: Session + RAG 결합

In [ ]:
# ============================================================
# 🔴 Quiz 5: RAG 진로상담 에이전트 직접 구현
# ============================================================
# TODO: ??? 부분을 채워서 완성하세요! (15분)
# ============================================================

# 1️⃣ load_chunks() 구현
def my_load_chunks(filepath: str, min_len: int = 80) -> list:
    with open(???, "r", encoding="utf-8") as f:
        text = ???
    sections = [s.strip() for s in text.split(???) if s.strip()]
    return [s for s in sections if len(s) > ???]

my_chunks = my_load_chunks("data/rag_source.md")
print(f"✅ {len(my_chunks)}개 청크")

# 2️⃣ TF-IDF 벡터화
my_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=5000)
my_vectors = my_vectorizer.fit_transform(my_chunks)

# 3️⃣ @tool 검색 함수 구현
@tool(approval_mode="never_require")
def my_search(query: str) -> str:
    """진로, 학습, AI 직업 정보를 검색합니다."""
    q_vec = my_vectorizer.transform([???])
    sims = cosine_similarity(???, ???)[0]
    top_idx = np.argsort(sims)[::-1][:3]
    results = [(my_chunks[i], float(sims[i])) for i in top_idx if sims[i] > 0]
    if not results:
        return "관련 정보를 찾지 못했습니다."
    parts = [f"[참고 자료 {i+1}]\n{chunk[:400]}" for i, (chunk, _) in enumerate(results)]
    return "\n\n".join(???)

# 4️⃣ RAG 에이전트 생성
my_rag = Agent(
    client=client,
    name="내진로상담AI",
    instructions="???",  # my_search 도구 사용 명시, 출처 표시 요구
    tools=[???]
)

print("✅ 나만의 RAG 에이전트 준비!")

# 5️⃣ 진로 질문 3개 실행!
my_questions = [
    "???",  # 예: "머신러닝 엔지니어가 되려면?"
    "???",  # 예: "수학을 못해도 AI 분야에 갈 수 있나요?"
    "???",  # 내가 정말 궁금한 것!
]

for i, q in enumerate(my_questions, 1):
    print(f"\n{'='*60}\n❓ Q{i}: {q}")
    ans = await my_rag.run(???)
    print(f"🤖 답변:\n{ans.text}")

In [ ]:
# 🌟 보너스: RAG + Session
bonus_session = my_rag.create_session()  # 동기!

bonus_talks = [
    "안녕! 나는 고2야. 장래희망이 클라우드 AI 아키텍트야.",
    "내 꿈 직업에 대해 좀 더 자세히 알려줘.",    # RAG 작동!
    "내 꿈이 뭐라고 했지?",                       # 메모리 확인!
]
for msg in bonus_talks:
    print(f"\n👤 나: {msg}")
    r = await my_rag.run(msg, ???=bonus_session)
    print(f"🤖 AI: {r.text}")

In [ ]:
# ✅ Quiz 5 정답 (강사 참고용)
# def my_load_chunks(filepath, min_len=80):
#     with open(filepath, "r", encoding="utf-8") as f:
#         text = f.read()
#     sections = [s.strip() for s in text.split("---") if s.strip()]
#     return [s for s in sections if len(s) > min_len]
#
# @tool(approval_mode="never_require")
# def my_search(query: str) -> str:
#     """진로, 학습, AI 직업 정보를 검색합니다."""
#     q_vec = my_vectorizer.transform([query])
#     sims = cosine_similarity(q_vec, my_vectors)[0]
#     top_idx = np.argsort(sims)[::-1][:3]
#     results = [(my_chunks[i], float(sims[i])) for i in top_idx if sims[i] > 0]
#     if not results: return "관련 정보를 찾지 못했습니다."
#     return "\n\n".join([f"[참고 자료 {i+1}]\n{c[:400]}" for i, (c, _) in enumerate(results)])
#
# my_rag = Agent(client=client, name="내진로상담AI",
#     instructions="당신은 고등학생 진로 상담 AI입니다. my_search 도구로 정보를 검색하고 [출처: 참고 자료 N]을 표시하세요.",
#     tools=[my_search])
# ans = await my_rag.run("AI 관련 직업에는 어떤 것들이 있나요?")
# print(ans.text)

---
# 🎤 마무리 & 해커톤 예고

## 오늘 배운 MAF 1.0.0 핵심 패턴 정리

| 패턴 | 클래스 | 언제 사용? | 예시 |
|------|--------|-----------|------|
| **Sequential** | `SequentialBuilder` | 단계별 파이프라인 | 작가→편집자→검토자 |
| **Concurrent** | `ConcurrentBuilder` | 독립적 병렬 분석 | 기술·시장·윤리 동시 분석 |
| **GroupChat** | `GroupChatBuilder` | 역할 토론/협업 | 찬성·반대·중재 토론 |
| **Session** | `create_session()` | 대화 기억 유지 | 공부 플래너 |
| **RAG + @tool** | `@tool` | 외부 지식 검색 | 진로 상담 |

## ⚠️ 오늘 꼭 기억할 API 주의사항

```python
# ✅ 올바른 GroupChatBuilder 사용법
workflow = GroupChatBuilder(
    participants=[a, b, c],      # 생성자에 직접!
    selection_func=my_func,
).build()

# ✅ GroupChatState 필드명
state.current_round   # int (라운드 번호)
state.participants    # OrderedDict (에이전트 이름 → 설명)
state.conversation    # list[Message]

# ✅ create_session()은 동기!
session = agent.create_session()  # await 없음!

# ✅ run() 결과에서 텍스트 추출
response = await agent.run("...", session=session)
print(response.text)              # .text 속성 사용
```

*🌲 PINE 2기 여러분, 오늘도 정말 수고했습니다!*